# The LaLonde benchmark: does job training raise earnings?

You shouldn't trust a causal estimation library at face value. This notebook takes the classic
benchmark for observational causal methods — the **National Supported Work Demonstration (NSW)**
data from LaLonde (1986) and Dehejia & Wahba (1999) — runs it through `formative`, and compares
the results against the published figures. The data is downloaded from its canonical public
source, so every number here is reproducible by running the notebook top to bottom.

**The setup.** The NSW was a 1970s US programme that gave disadvantaged workers 9–18 months of
subsidised work experience. Admission was **randomised**, so the programme doubles as an RCT: we
know the true effect of the training on 1978 earnings. LaLonde's famous test threw away the
experimental control group, replaced it with survey respondents from the Panel Study of Income
Dynamics (PSID) and the Current Population Survey (CPS), and asked whether econometric methods
could recover the experimental answer from observational data. Naive comparisons were wildly
wrong; Dehejia & Wahba later showed propensity score methods land close to the benchmark.

This lets us check three things at once:

1. Does `RCT` reproduce the published experimental effect (**\$1,794** in the Dehejia-Wahba sample)?
2. Does the naive comparison on observational data go as badly wrong as LaLonde reported
   (**−\$15,205** against PSID controls)?
3. Does `PropensityScoreMatching` pull the estimate back to the neighbourhood of the
   experimental answer, as Dehejia & Wahba found?

## Loading the data

The Dehejia-Wahba sample is hosted on [Rajeev Dehejia's data page](https://users.nber.org/~rdehejia/nswdata2.html).
Each file has the same ten columns: a treatment indicator, pre-treatment characteristics (age,
education, race, marital status, and real earnings in 1974 and 1975), and the outcome — real
earnings in 1978 (`re78`).

In [16]:
import pandas as pd

BASE = "https://users.nber.org/~rdehejia/data"
COLS = [
    "treat", "age", "education", "black", "hispanic",
    "married", "nodegree", "re74", "re75", "re78",
]


def load(name):
    return pd.read_csv(f"{BASE}/{name}.txt", sep=r"\s+", header=None, names=COLS)


treated = load("nswre74_treated")               # 185 NSW participants
experimental_control = load("nswre74_control")  # 260 randomised controls
psid = load("psid_controls")                    # 2,490 PSID respondents (never in NSW)
cps = load("cps_controls")                      # 15,992 CPS respondents (never in NSW)

experimental = pd.concat([treated, experimental_control], ignore_index=True)
observational_psid = pd.concat([treated, psid], ignore_index=True)
observational_cps = pd.concat([treated, cps], ignore_index=True)

print(f"experimental sample : N={len(experimental)}")
print(f"treated + PSID      : N={len(observational_psid)}")
print(f"treated + CPS       : N={len(observational_cps)}")

experimental sample : N=445
treated + PSID      : N=2675
treated + CPS       : N=16177


## Step 1: the experimental answer

Because NSW admission was randomised, the DAG is as simple as it gets — treatment causes the
outcome, and nothing causes treatment:

In [17]:
from formative.causal import DAG, RCT

rct_dag = DAG()
rct_dag.assume("treat").causes("re78")

rct_result = RCT(rct_dag, treatment="treat", outcome="re78").fit(experimental)
print(rct_result.summary())


RCT Causal Effect: treat → re78
  Estimand: ATE (average treatment effect)
──────────────────────────────────────────────────
  ATE estimate         :  1794.3424

  Std. error           :   632.8534
  95% CI               : [550.5745, 3038.1103]
  p-value              :     0.0048
  N                    :        445

  Assumptions
  ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄
  [ untestable ]  Random assignment of treatment
  [ untestable ]  Excludability: assignment affects outcome only through treatment received
  [ untestable ]  Stable Unit Treatment Value Assumption (SUTVA)



**\$1,794.34** — the published experimental benchmark for this sample is \$1,794 (Dehejia &
Wahba 1999, Table 3). This is the ground truth the observational methods below will be judged
against.

## Step 2: the naive comparison goes badly wrong

Now we discard the experimental controls and compare NSW participants against PSID respondents,
as LaLonde did. The comparison group is now nothing like the treated group — PSID respondents
are older, better educated, far more likely to be employed — so the covariates confound the
treatment/outcome relationship. The observational DAG declares this:

In [18]:
from formative.causal import OLSObservational

COVARIATES = ["age", "education", "black", "hispanic",
              "married", "nodegree", "re74", "re75"]

obs_dag = DAG()
for c in COVARIATES:
    obs_dag.assume(c).causes("treat", "re78")
obs_dag.assume("treat").causes("re78")

ols_result = OLSObservational(obs_dag, treatment="treat", outcome="re78").fit(observational_psid)
print(ols_result.summary())


OLS Causal Effect: treat → re78
──────────────────────────────────────────────────
  Adjusted estimate    :   751.9464  (controlling for: age, black, education, hispanic, married, nodegree, re74, re75)
  Unadjusted estimate  : -15204.7775  (no controls)
  Confounding bias     : -15956.7239

  Std. error           :   915.2572
  95% CI               : [-1042.7399, 2546.6327]
  p-value              :     0.4114
  N                    :       2675

  Assumptions
  ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄
  [ untestable ]  No unobserved confounders (selection on observables)
  [ untestable ]  Correct functional form for control variables
  [ untestable ]  No reverse causality
  [ untestable ]  No measurement error in key variables
  [ untestable ]  Stable Unit Treatment Value Assumption (SUTVA)



This single summary reproduces LaLonde's core finding. The unadjusted estimate — the answer you
would get by simply comparing average earnings between the two groups — is **−\$15,204.78**,
matching the −\$15,205 raw gap in the published data. Taken at face value it says the training
programme *destroyed* fifteen thousand dollars of annual earnings; in reality it measures how
much poorer NSW participants were than average PSID respondents to begin with. Linear regression
adjustment moves the estimate to \$752 — the right sign, but still less than half the
experimental benchmark and statistically indistinguishable from zero, echoing LaLonde's
conclusion that regression on a badly mismatched comparison group cannot be trusted. The
adjusted-vs-unadjusted comparison that every `formative` result carries exists precisely to make
this kind of confounding visible.

## Step 3: matching recovers the benchmark

Dehejia & Wahba's insight was that most PSID respondents are so unlike NSW participants that a
linear model extrapolates across incomparable people. Propensity score matching instead compares
each treated person only with the most similar controls:

In [19]:
from formative.causal import PropensityScoreMatching

psm_psid = PropensityScoreMatching(obs_dag, treatment="treat", outcome="re78").fit(observational_psid)
print(psm_psid.summary())


PSM Causal Effect: treat → re78
  Estimand: ATT (average treatment effect on the treated)
──────────────────────────────────────────────────────
  ATT estimate         :  2125.7131  (controlling for: age, black, education, hispanic, married, nodegree, re74, re75)
  Unadjusted estimate  : -15204.7775  (naive mean difference)
  Confounding bias     : -17330.4906

  Std. error           :  1079.2522  (bootstrap, N=500)
  95% CI               : [-943.4393, 3438.7593]  (bootstrap percentile)
  p-value              :     0.0489
  N                    :       2675

  Matching: 1-to-1 nearest-neighbour on propensity score (with replacement)

  Assumptions
  ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄
  [ untestable ]  Conditional independence: no unobserved confounders given matched variables
  [  testable  ]  Common support: overlap exists in characteristics between groups
  [ untestable ]  Correct specification of the matching variables
  [ untestable ]  Stable Unit Treatment Value Ass

From the same observational data that produced −\$15,205 naively, matching estimates
**\$2,125.71** — within one standard error of the experimental \$1,794. As a robustness check,
we repeat the exercise with the much larger CPS comparison group (this bootstraps matching over
16,177 rows, so it takes a minute or two):

In [20]:
psm_cps = PropensityScoreMatching(obs_dag, treatment="treat", outcome="re78").fit(observational_cps)
print(psm_cps.summary())


PSM Causal Effect: treat → re78
  Estimand: ATT (average treatment effect on the treated)
──────────────────────────────────────────────────────
  ATT estimate         :  2074.0167  (controlling for: age, black, education, hispanic, married, nodegree, re74, re75)
  Unadjusted estimate  : -8497.5161  (naive mean difference)
  Confounding bias     : -10571.5328

  Std. error           :   922.1399  (bootstrap, N=500)
  95% CI               : [-270.7255, 3332.0155]  (bootstrap percentile)
  p-value              :     0.0245
  N                    :      16177

  Matching: 1-to-1 nearest-neighbour on propensity score (with replacement)

  Assumptions
  ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄
  [ untestable ]  Conditional independence: no unobserved confounders given matched variables
  [  testable  ]  Common support: overlap exists in characteristics between groups
  [ untestable ]  Correct specification of the matching variables
  [ untestable ]  Stable Unit Treatment Value Assu

Two very different comparison groups, matched estimates a mere \$50 apart, both bracketing the
experimental truth. Dehejia & Wahba's own matching estimates were \$1,691 (PSID) and \$1,582
(CPS); the residual spread between their numbers and ours reflects implementation choices (they
used stratification and different matching variants; `formative` uses 1-to-1 nearest-neighbour
matching with replacement), and later work (Smith & Todd 2005) showed estimates in this range
are sensitive to exactly such choices. All of them are within one standard error of the
experimental answer.

## Scorecard

In [21]:
scorecard = pd.DataFrame(
    [
        ("Experimental ATE (RCT)", 1794, rct_result.effect),
        ("Naive mean difference, PSID controls", -15205, ols_result.unadjusted_effect),
        ("Naive mean difference, CPS controls", -8498, psm_cps.unadjusted_effect),
        ("PSM ATT, PSID controls (DW 1999: $1,691)", 1691, psm_psid.effect),
        ("PSM ATT, CPS controls (DW 1999: $1,582)", 1582, psm_cps.effect),
    ],
    columns=["Quantity", "Published", "formative"],
)
scorecard.round(2)

,Quantity,Published,formative
0,Experimental ATE (RCT),1794,1794.34
1,"Naive mean difference, PSID controls",-15205,-15204.78
2,"Naive mean difference, CPS controls",-8498,-8497.52
3,"PSM ATT, PSID controls (DW 1999: $1,691)",1691,2125.71
4,"PSM ATT, CPS controls (DW 1999: $1,582)",1582,2074.02


The pattern is exactly what forty years of literature says it should be: the experimental
estimate is reproduced to the dollar, the naive observational comparison is catastrophically
biased (and `formative`'s unadjusted estimate quantifies that bias to the dollar), and
propensity score matching recovers an estimate statistically indistinguishable from the
experimental truth.

## References

- LaLonde, R. (1986). "Evaluating the Econometric Evaluations of Training Programs with
  Experimental Data." *American Economic Review*, 76(4), 604–620.
- Dehejia, R. & Wahba, S. (1999). "Causal Effects in Nonexperimental Studies: Reevaluating the
  Evaluation of Training Programs." *Journal of the American Statistical Association*, 94(448),
  1053–1062.
- Smith, J. & Todd, P. (2005). "Does matching overcome LaLonde's critique of nonexperimental
  estimators?" *Journal of Econometrics*, 125(1–2), 305–353.
- Data: [Rajeev Dehejia's NSW data page](https://users.nber.org/~rdehejia/nswdata2.html).